# Contribution A Real Active-Round Pilot

This notebook runs a real one-round M-OWODB Task-1 to Task-2 integration pilot with the actual PROB detector. It compares `random` and `full` acquisition from the same Task-1 checkpoint, source split, controlled long-tail candidate pool, reference IDs, seed, budget, training settings, and evaluation split.

The pilot exercises the end-to-end path: controlled Task-2 pool construction, detector proposal export, distribution-aware scoring, fixed-budget selection, complete-image annotation reveal, one-epoch PROB retraining, official PROB evaluation, and a compact comparison table. It is an integration experiment, not a full benchmark result.

In [ ]:
# User-editable configuration.
SEED = 0
BUDGET = 10
SOURCE_TASK2_IMAGES = 300
REFERENCE_IMAGES = 30
EVAL_UNKNOWN_IMAGES = 100
EVAL_KNOWN_IMAGES = 100
IMBALANCE_RATIO = 20.0
TRAIN_EPOCHS = 1
BATCH_SIZE = 1
NUM_WORKERS = 2
STRATEGIES = ("random", "full")

# None means: load the existing default from configs/experiment.yaml.
ALPHA = None
BETA = None
GAMMA = None
COHERENCE_POWER = None
TOP_K = None

DAOWOD_REPOSITORY_URL = "https://github.com/gubiczam/distribution-aware-owod.git"
DAOWOD_BRANCH = "main"
PROB_REPOSITORY_URL = "https://github.com/gubiczam/PROB.git"
PROB_BRANCH = "feat/daowod-bridge"

DRIVE_ROOT = "/content/drive/MyDrive/DAOWOD"
DRIVE_ARCHIVE = f"{DRIVE_ROOT}/assets/OWOD_full.tar.zst"
DRIVE_TASK1_CHECKPOINT = f"{DRIVE_ROOT}/checkpoints/MOWODB/t1.pth"
DRIVE_RESULT_DIR = f"{DRIVE_ROOT}/results/real_active_round_pilot"

CONTENT_ROOT = "/content"
DAOWOD_PATH = f"{CONTENT_ROOT}/distribution-aware-owod"
PROB_PATH = f"{CONTENT_ROOT}/PROB"
DATA_ROOT = f"{CONTENT_ROOT}/data/OWOD"
LOCAL_RESULT_ROOT = f"{CONTENT_ROOT}/real_active_round_pilot"

DATASET = "TOWOD"
TASK2_SOURCE_SPLIT = "owod_t2_train"
TASK1_REFERENCE_SPLIT = "owod_t1_train"
OFFICIAL_EVAL_SPLIT = "owod_all_task_test"
PILOT_EVAL_SPLIT = "daowod_pilot_balanced_test"
PREVIOUS_CLASSES = 20
CURRENT_CLASSES = 20
NUM_CLASSES = 81
MAX_PROPOSALS_PER_IMAGE = 20
MINIMUM_UNKNOWN_SCORE = 0.0
DEVICE = "cuda"

STATUS = {
    "GPU": "PENDING",
    "Drive assets": "PENDING",
    "repositories": "PENDING",
    "DAOWOD validation": "PENDING",
    "PROB bridge": "PENDING",
    "attention backend": "PENDING",
    "protocol construction": "PENDING",
    "selective extraction": "PENDING",
    "random round": "PENDING",
    "full round": "PENDING",
    "Drive persistence": "PENDING",
}
STATUS_ORDER = tuple(STATUS)


In [ ]:
import json
import platform
import random
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

import pandas as pd


def run_cmd(command, *, cwd=None, timeout=600, check=True):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout[-8000:])
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result


def run_step(name, function):
    print(f"\n=== {name} ===")
    try:
        value = function()
    except Exception:
        STATUS[name] = "FAILED"
        print(f"FAILED stage: {name}")
        traceback.print_exc()
        raise
    STATUS[name] = "OK"
    return value


def read_ids(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing image set: {path}")
    return [line.split()[0] for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_ids(path, image_ids):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(image_ids) + ("\n" if image_ids else ""), encoding="utf-8")


def disk_free(path="/content"):
    return round(shutil.disk_usage(path).free / 1024**3, 2)


def runtime_and_drive():
    import torch
    from google.colab import drive

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required. Select a Colab T4 GPU runtime.")
    drive.mount("/content/drive", force_remount=False)
    archive = Path(DRIVE_ARCHIVE)
    checkpoint = Path(DRIVE_TASK1_CHECKPOINT)
    missing = [str(path) for path in (archive, checkpoint) if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing Drive asset(s): " + ", ".join(missing))
    rows = {
        "Python": sys.version.split()[0],
        "Platform": platform.platform(),
        "PyTorch": torch.__version__,
        "CUDA": torch.version.cuda,
        "GPU": torch.cuda.get_device_name(0),
        "Archive": str(archive),
        "Task-1 checkpoint": str(checkpoint),
        "Free /content GB": disk_free("/content"),
    }
    display(pd.DataFrame(rows.items(), columns=["item", "value"]))
    return rows

runtime_info = run_step("GPU", runtime_and_drive)
STATUS["Drive assets"] = "OK"


In [ ]:
import importlib.util


def clone_and_validate_repositories():
    for name in ("distribution-aware-owod", "PROB"):
        checkout = Path(CONTENT_ROOT) / name
        if checkout.exists():
            shutil.rmtree(checkout)

    run_cmd(["git", "clone", "--branch", DAOWOD_BRANCH, "--single-branch", DAOWOD_REPOSITORY_URL, DAOWOD_PATH], cwd=CONTENT_ROOT, timeout=300)
    run_cmd(["git", "clone", "--branch", PROB_BRANCH, "--single-branch", PROB_REPOSITORY_URL, PROB_PATH], cwd=CONTENT_ROOT, timeout=300)
    run_cmd([sys.executable, "-m", "pip", "install", "--editable", ".[dev]"], cwd=DAOWOD_PATH, timeout=900)

    prob_imports = {
        "wandb": "wandb",
        "einops": "einops",
        "pycocotools": "pycocotools",
        "skimage": "scikit-image",
        "joblib": "joblib",
        "tqdm": "tqdm",
        "ipdb": "ipdb",
    }
    missing = [package for module, package in prob_imports.items() if importlib.util.find_spec(module) is None]
    if missing:
        run_cmd([sys.executable, "-m", "pip", "install", *missing], cwd=PROB_PATH, timeout=900)
    else:
        print("PROB dependency imports already available; no PROB pip install needed.")

    run_cmd(["ruff", "check", "."], cwd=DAOWOD_PATH, timeout=300)
    run_cmd(["pytest", "-q"], cwd=DAOWOD_PATH, timeout=300)
    run_cmd([sys.executable, "-m", "compileall", "-q", "src", "tests"], cwd=DAOWOD_PATH, timeout=300)
    STATUS["DAOWOD validation"] = "OK"

    run_cmd([sys.executable, "daowod_prob_bridge.py", "check"], cwd=PROB_PATH, timeout=120)
    STATUS["PROB bridge"] = "OK"

    commits = {
        "DAOWOD": run_cmd(["git", "rev-parse", "HEAD"], cwd=DAOWOD_PATH).stdout.strip(),
        "PROB": run_cmd(["git", "rev-parse", "HEAD"], cwd=PROB_PATH).stdout.strip(),
    }
    display(pd.DataFrame(commits.items(), columns=["repository", "commit"]))
    return commits

repo_commits = run_step("repositories", clone_and_validate_repositories)


In [ ]:
def patch_attention_fallback():
    func_path = Path(PROB_PATH) / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
    module_path = Path(PROB_PATH) / "models" / "ops" / "modules" / "ms_deform_attn.py"
    func_source = func_path.read_text(encoding="utf-8")
    if "MSDA = None" not in func_source:
        func_source = func_source.replace(
            "import MultiScaleDeformableAttention as MSDA",
            "try:\n    import MultiScaleDeformableAttention as MSDA\nexcept ModuleNotFoundError:\n    MSDA = None",
        )
        func_path.write_text(func_source, encoding="utf-8")
    module_source = module_path.read_text(encoding="utf-8")
    module_source = module_source.replace(
        "from ..functions import MSDeformAttnFunction",
        "from ..functions.ms_deform_attn_func import MSDeformAttnFunction, ms_deform_attn_core_pytorch",
    )
    old = "output = MSDeformAttnFunction.apply(\n            value, input_spatial_shapes, input_level_start_index, sampling_locations, attention_weights, self.im2col_step)"
    new = "output = ms_deform_attn_core_pytorch(\n            value, input_spatial_shapes, sampling_locations, attention_weights)"
    if old in module_source:
        module_source = module_source.replace(old, new)
        module_path.write_text(module_source, encoding="utf-8")


def attention_forward_backward_test():
    code = "\n".join([
        "import torch",
        "from models.ops.modules.ms_deform_attn import MSDeformAttn",
        "assert torch.cuda.is_available()",
        "device = torch.device('cuda')",
        "module = MSDeformAttn(d_model=8, n_levels=1, n_heads=2, n_points=2).to(device)",
        "query = torch.randn(1, 3, 8, device=device, requires_grad=True)",
        "reference_points = torch.full((1, 3, 1, 2), 0.5, device=device)",
        "input_flatten = torch.randn(1, 4, 8, device=device, requires_grad=True)",
        "input_spatial_shapes = torch.tensor([[2, 2]], dtype=torch.long, device=device)",
        "input_level_start_index = torch.tensor([0], dtype=torch.long, device=device)",
        "output = module(query, reference_points, input_flatten, input_spatial_shapes, input_level_start_index)",
        "loss = output.square().mean()",
        "loss.backward()",
        "assert output.shape == (1, 3, 8)",
        "assert query.grad is not None and torch.isfinite(query.grad).all()",
        "assert input_flatten.grad is not None and torch.isfinite(input_flatten.grad).all()",
        "print('attention forward/backward OK', tuple(output.shape))",
    ])
    return run_cmd([sys.executable, "-c", code], cwd=PROB_PATH, timeout=180, check=False)


def ensure_main_open_world_fix():
    path = Path(PROB_PATH) / "main_open_world.py"
    source = path.read_text(encoding="utf-8")
    marker = "    print(f'Start training from epoch {args.start_epoch} to {args.epochs}')"
    if "coco_evaluator = None" not in source:
        if marker not in source:
            raise RuntimeError("Could not locate the main_open_world.py training-loop marker.")
        source = source.replace(marker, "    test_stats = {}\n    coco_evaluator = None\n\n" + marker)
        path.write_text(source, encoding="utf-8")
    updated = path.read_text(encoding="utf-8")
    if "test_stats = {}" not in updated or "coco_evaluator = None" not in updated:
        raise RuntimeError("main_open_world.py does not contain the test_stats initialization fix.")


def wrap_task1_checkpoint():
    wrapper_path = Path(CONTENT_ROOT) / "t1_epoch40.pth"
    code = "\n".join([
        "from pathlib import Path",
        "import torch",
        f"source = Path({DRIVE_TASK1_CHECKPOINT!r})",
        f"target = Path({str(wrapper_path)!r})",
        "try:",
        "    checkpoint = torch.load(source, map_location='cpu', weights_only=False)",
        "except TypeError:",
        "    checkpoint = torch.load(source, map_location='cpu')",
        "if isinstance(checkpoint, dict) and isinstance(checkpoint.get('epoch'), int):",
        "    target.write_bytes(source.read_bytes())",
        "elif isinstance(checkpoint, dict) and 'model' in checkpoint:",
        "    checkpoint = dict(checkpoint)",
        "    checkpoint['epoch'] = 40",
        "    torch.save(checkpoint, target)",
        "else:",
        "    torch.save({'model': checkpoint, 'epoch': 40}, target)",
        "print(target)",
    ])
    run_cmd([sys.executable, "-c", code], cwd=PROB_PATH, timeout=300)
    return wrapper_path


def prepare_prob_runtime():
    ops_dir = Path(PROB_PATH) / "models" / "ops"
    build = run_cmd([sys.executable, "setup.py", "build", "install"], cwd=ops_dir, timeout=900, check=False)
    attention_backend = "cuda-extension"
    test = attention_forward_backward_test()
    if build.returncode != 0 or test.returncode != 0:
        print("Using pure-PyTorch deformable-attention fallback after extension build/test failure.")
        patch_attention_fallback()
        attention_backend = "pytorch-fallback"
        test = attention_forward_backward_test()
        if test.returncode != 0:
            raise RuntimeError("Attention fallback forward/backward test failed.")

    dino_path = Path(PROB_PATH) / "models" / "dino_resnet50_pretrain.pth"
    if not dino_path.exists():
        run_cmd([sys.executable, "-c", "import urllib.request; urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth', 'models/dino_resnet50_pretrain.pth')"], cwd=PROB_PATH, timeout=600)
    checkpoint = wrap_task1_checkpoint()
    ensure_main_open_world_fix()
    rows = {"attention_backend": attention_backend, "dino_checkpoint": str(dino_path), "task1_checkpoint_for_prob": str(checkpoint), "main_open_world_fix": "OK"}
    display(pd.DataFrame(rows.items(), columns=["item", "value"]))
    return rows

prob_runtime = run_step("attention backend", prepare_prob_runtime)
TASK1_CHECKPOINT_FOR_PROB = prob_runtime["task1_checkpoint_for_prob"]
ATTENTION_BACKEND = prob_runtime["attention_backend"]


In [ ]:
def extract_imagesets_and_annotations():
    data_parent = Path(DATA_ROOT).parent
    data_parent.mkdir(parents=True, exist_ok=True)
    run_cmd([
        "tar", "--zstd", "-xf", DRIVE_ARCHIVE,
        "-C", str(data_parent),
        "OWOD/ImageSets", "OWOD/Annotations",
    ], timeout=1800)
    required = [
        Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK2_SOURCE_SPLIT}.txt",
        Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK1_REFERENCE_SPLIT}.txt",
        Path(DATA_ROOT) / "ImageSets" / DATASET / f"{OFFICIAL_EVAL_SPLIT}.txt",
        Path(DATA_ROOT) / "Annotations",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Archive did not provide required protocol files: " + ", ".join(missing))
    display(pd.DataFrame({"extracted": ["ImageSets", "Annotations"], "root": DATA_ROOT}))

extract_imagesets_and_annotations()


In [ ]:
import xml.etree.ElementTree as ET

sys.path.insert(0, PROB_PATH)
from datasets.torchvision_datasets.open_world import (  # noqa: E402
    BASE_VOC_CLASS_NAMES,
    T2_CLASS_NAMES,
    VOC_CLASS_NAMES_COCOFIED,
    VOC_COCO_CLASS_NAMES,
)
from daowod.dataset import build_long_tail_pool  # noqa: E402


def seeded_order_preserving_sample(image_ids, count, *, seed, salt):
    if count > len(image_ids):
        raise ValueError(f"Requested {count} images from only {len(image_ids)} IDs.")
    rng = random.Random(f"{seed}:{salt}")
    selected = set(rng.sample(list(image_ids), count))
    return [image_id for image_id in image_ids if image_id in selected]


def construct_training_protocol():
    image_set_root = Path(DATA_ROOT) / "ImageSets" / DATASET
    official_task2 = read_ids(image_set_root / f"{TASK2_SOURCE_SPLIT}.txt")
    task2_source_ids = seeded_order_preserving_sample(official_task2, SOURCE_TASK2_IMAGES, seed=SEED, salt="task2-source")
    pilot_source_split = image_set_root / "daowod_pilot_t2_source_train.txt"
    write_ids(pilot_source_split, task2_source_ids)
    long_tail_dir = Path(LOCAL_RESULT_ROOT) / "protocol" / "long_tail"
    pool = build_long_tail_pool(
        annotation_dir=Path(DATA_ROOT) / "Annotations",
        source_split=pilot_source_split,
        task_class_names=list(T2_CLASS_NAMES),
        output_dir=long_tail_dir,
        imbalance_ratio=IMBALANCE_RATIO,
        seed=SEED,
    )
    candidate_ids = list(pool["selected_image_ids"])
    if len(candidate_ids) < BUDGET:
        raise RuntimeError(f"Candidate pool has {len(candidate_ids)} images, below budget {BUDGET}.")
    official_t1 = read_ids(image_set_root / f"{TASK1_REFERENCE_SPLIT}.txt")
    t1_available = [image_id for image_id in official_t1 if image_id not in set(candidate_ids)]
    reference_ids = seeded_order_preserving_sample(t1_available, REFERENCE_IMAGES, seed=SEED, salt="reference")
    labelled_ids = []
    manifest = {
        "seed": SEED,
        "source_split": str(pilot_source_split),
        "source_image_count": len(task2_source_ids),
        "candidate_pool_count": len(candidate_ids),
        "reference_count": len(reference_ids),
        "initial_labelled_count": len(labelled_ids),
        "imbalance_ratio": IMBALANCE_RATIO,
        "candidate_ground_truth_visible_to_acquisition": False,
    }
    manifest_path = Path(LOCAL_RESULT_ROOT) / "protocol" / "training_protocol.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    display(pd.DataFrame(manifest.items(), columns=["item", "value"]))
    return candidate_ids, reference_ids, labelled_ids, str(pilot_source_split), str(pool["manifest_path"])

candidate_ids, reference_ids, labelled_ids, pilot_source_split, long_tail_manifest = construct_training_protocol()


In [ ]:
def annotation_classes(image_id):
    path = Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml"
    root = ET.parse(path).getroot()
    classes = []
    for node in root.findall("./object/name"):
        if node.text:
            name = node.text.strip()
            if name in VOC_CLASS_NAMES_COCOFIED:
                name = BASE_VOC_CLASS_NAMES[VOC_CLASS_NAMES_COCOFIED.index(name)]
            classes.append(name)
    return classes


def construct_evaluation_protocol():
    class_names = list(VOC_COCO_CLASS_NAMES[DATASET])
    class_to_index = {name: index for index, name in enumerate(class_names)}
    official_ids = read_ids(Path(DATA_ROOT) / "ImageSets" / DATASET / f"{OFFICIAL_EVAL_SPLIT}.txt")
    unknown_candidates = []
    known_candidates = []
    for image_id in official_ids:
        indices = [class_to_index[name] for name in annotation_classes(image_id) if name in class_to_index]
        if not indices:
            continue
        has_unknown = any(40 <= index <= 79 for index in indices)
        known_only = all(0 <= index <= 39 for index in indices)
        if has_unknown:
            unknown_candidates.append(image_id)
        elif known_only:
            known_candidates.append(image_id)
    unknown_ids = seeded_order_preserving_sample(unknown_candidates, EVAL_UNKNOWN_IMAGES, seed=SEED, salt="eval-unknown")
    known_ids = seeded_order_preserving_sample(known_candidates, EVAL_KNOWN_IMAGES, seed=SEED, salt="eval-known")
    selected = set(unknown_ids) | set(known_ids)
    evaluation_ids = [image_id for image_id in official_ids if image_id in selected]
    split_path = Path(DATA_ROOT) / "ImageSets" / DATASET / f"{PILOT_EVAL_SPLIT}.txt"
    write_ids(split_path, evaluation_ids)
    manifest = {
        "seed": SEED,
        "official_source_split": OFFICIAL_EVAL_SPLIT,
        "pilot_split": PILOT_EVAL_SPLIT,
        "known_class_indices": "0..39",
        "unknown_class_indices": "40..79",
        "unknown_image_count": len(unknown_ids),
        "known_only_image_count": len(known_ids),
        "evaluation_count": len(evaluation_ids),
        "ground_truth_use": "evaluation_subset_construction_only",
    }
    manifest_path = Path(LOCAL_RESULT_ROOT) / "protocol" / "evaluation_protocol.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    display(pd.DataFrame(manifest.items(), columns=["item", "value"]))
    return evaluation_ids, str(split_path), str(manifest_path)

evaluation_ids, evaluation_split_path, evaluation_manifest_path = construct_evaluation_protocol()
STATUS["protocol construction"] = "OK"


In [ ]:
def extract_required_jpegs():
    required_ids = list(dict.fromkeys([*candidate_ids, *reference_ids, *evaluation_ids]))
    files_list = Path(CONTENT_ROOT) / "daowod_required_jpegs.txt"
    files_list.write_text(
        "\n".join(f"OWOD/JPEGImages/{image_id}.jpg" for image_id in required_ids) + "\n",
        encoding="utf-8",
    )
    run_cmd([
        "tar", "--zstd", "-xf", DRIVE_ARCHIVE,
        "-C", str(Path(DATA_ROOT).parent),
        "--files-from", str(files_list),
    ], timeout=3600)
    missing = []
    for image_id in required_ids:
        for relative in (f"JPEGImages/{image_id}.jpg", f"Annotations/{image_id}.xml"):
            path = Path(DATA_ROOT) / relative
            if not path.exists():
                missing.append(str(path))
    if missing:
        raise FileNotFoundError("Selective extraction missing required files: " + ", ".join(missing[:20]))
    rows = {
        "candidate count": len(candidate_ids),
        "reference count": len(reference_ids),
        "evaluation count": len(evaluation_ids),
        "total extracted image count": len(required_ids),
        "free /content GB": disk_free("/content"),
    }
    display(pd.DataFrame(rows.items(), columns=["item", "value"]))
    return required_ids

required_image_ids = run_step("selective extraction", extract_required_jpegs)


In [ ]:
from daowod import ProbAdapter, load_config, run_active_round  # noqa: E402
from daowod.config import AcquisitionConfig, AcquisitionWeights  # noqa: E402

base_config = load_config(Path(DAOWOD_PATH) / "configs" / "experiment.yaml")
base_weights = base_config.acquisition.weights
weights = AcquisitionWeights(
    uncertainty=base_weights.uncertainty if ALPHA is None else ALPHA,
    novelty=base_weights.novelty if BETA is None else BETA,
    rarity=base_weights.rarity if GAMMA is None else GAMMA,
    coherence_power=base_weights.coherence_power if COHERENCE_POWER is None else COHERENCE_POWER,
    rarity_power=base_weights.rarity_power,
)
acquisition_config = AcquisitionConfig(
    strategies=STRATEGIES,
    uncertainty_mode=base_config.acquisition.uncertainty_mode,
    pseudo_label_source=base_config.acquisition.pseudo_label_source,
    cluster_count=base_config.acquisition.cluster_count,
    neighbour_count=base_config.acquisition.neighbour_count,
    top_k=base_config.acquisition.top_k if TOP_K is None else TOP_K,
    weights=weights,
)
common_args = (
    f"--data-root {DATA_ROOT} --dataset {DATASET} "
    f"--prev-introduced-classes {PREVIOUS_CLASSES} --current-introduced-classes {CURRENT_CLASSES} "
    f"--num-classes {NUM_CLASSES} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} "
    f"--device {DEVICE} --seed {SEED}"
)
adapter = ProbAdapter(
    repository_path=PROB_PATH,
    timeout_seconds=86400,
    train_command=(
        "python daowod_prob_bridge.py train "
        "--labelled-ids {labelled_ids} --previous-checkpoint {previous_checkpoint} "
        "--output-checkpoint {checkpoint} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} --epochs {TRAIN_EPOCHS} {common_args}"
    ),
    predict_command=(
        "python daowod_prob_bridge.py predict "
        "--image-ids {image_ids} --checkpoint {checkpoint} --output {proposals} "
        f"--max-proposals-per-image {MAX_PROPOSALS_PER_IMAGE} "
        f"--minimum-unknown-score {MINIMUM_UNKNOWN_SCORE} {common_args}"
    ),
    evaluate_command=(
        "python daowod_prob_bridge.py evaluate "
        "--checkpoint {checkpoint} --output {metrics} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} {common_args}"
    ),
)
configuration_rows = {
    "seed": SEED,
    "budget": BUDGET,
    "source task2 images": SOURCE_TASK2_IMAGES,
    "candidate pool count": len(candidate_ids),
    "reference images": REFERENCE_IMAGES,
    "evaluation images": len(evaluation_ids),
    "imbalance ratio": IMBALANCE_RATIO,
    "train epochs": TRAIN_EPOCHS,
    "batch size": BATCH_SIZE,
    "workers": NUM_WORKERS,
    "strategies": ", ".join(STRATEGIES),
    "alpha": acquisition_config.weights.uncertainty,
    "beta": acquisition_config.weights.novelty,
    "gamma": acquisition_config.weights.rarity,
    "coherence power": acquisition_config.weights.coherence_power,
    "top k": acquisition_config.top_k,
}
display(pd.DataFrame(configuration_rows.items(), columns=["setting", "value"]))


In [ ]:
experiment_rows = [
    {
        "strategy": strategy,
        "checkpoint": TASK1_CHECKPOINT_FOR_PROB,
        "candidate_ids": len(candidate_ids),
        "reference_ids": len(reference_ids),
        "initial_labelled_ids": len(labelled_ids),
        "budget": BUDGET,
        "seed": SEED,
        "evaluation_split": PILOT_EVAL_SPLIT,
        "output_dir": str(Path(LOCAL_RESULT_ROOT) / strategy),
    }
    for strategy in STRATEGIES
]
display(pd.DataFrame(experiment_rows))

round_results = {}
for strategy in STRATEGIES:
    output_dir = Path(LOCAL_RESULT_ROOT) / strategy
    manifest_path = output_dir / "round_manifest.json"
    if manifest_path.exists() and json.loads(manifest_path.read_text(encoding="utf-8")).get("completed") is True:
        raise RuntimeError(f"Completed round already exists and will not be overwritten: {output_dir}")
    result = run_active_round(
        adapter=adapter,
        checkpoint=TASK1_CHECKPOINT_FOR_PROB,
        candidate_ids=candidate_ids,
        reference_ids=reference_ids,
        labelled_ids=labelled_ids,
        output_dir=output_dir,
        strategy=strategy,
        budget=BUDGET,
        acquisition_config=acquisition_config,
        seed=SEED,
        round_index=0,
    )
    round_results[strategy] = result
    STATUS[f"{strategy} round"] = "OK"


In [ ]:
import numpy as np

REQUIRED_METRICS = {"known_mAP", "U_Recall", "WI", "A_OSE"}


def load_round(strategy):
    directory = Path(LOCAL_RESULT_ROOT) / strategy
    metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
    manifest = json.loads((directory / "round_manifest.json").read_text(encoding="utf-8"))
    selected = read_ids(directory / "selected_ids.txt")
    remaining = read_ids(directory / "remaining_pool_ids.txt")
    return directory, metrics, manifest, selected, remaining

round_data = {strategy: load_round(strategy) for strategy in STRATEGIES}
for strategy, (directory, metrics, manifest, selected, remaining) in round_data.items():
    assert manifest["completed"] is True, f"{strategy} did not complete"
    assert manifest["input_checkpoint"] == TASK1_CHECKPOINT_FOR_PROB
    assert manifest["budget"] == BUDGET
    assert manifest["seed"] == SEED
    assert len(selected) == BUDGET
    assert REQUIRED_METRICS <= set(metrics), f"{strategy} metrics missing required fields"
    assert set(selected) <= set(candidate_ids), f"{strategy} selected outside candidate pool"
    assert set(selected).isdisjoint(remaining), f"{strategy} selected IDs still remain in pool"
    assert (directory / "checkpoint.pth").exists(), f"{strategy} checkpoint missing"

candidate_proposal_image_ids = []
for strategy in STRATEGIES:
    with np.load(Path(LOCAL_RESULT_ROOT) / strategy / "candidate_proposals.npz", allow_pickle=True) as proposals:
        proposal_ids = [str(value) for value in proposals["image_ids"].tolist()]
    assert set(proposal_ids) == set(candidate_ids)
    candidate_proposal_image_ids.append(proposal_ids)
assert candidate_proposal_image_ids[0] == candidate_proposal_image_ids[1], "Candidate proposal order differs across strategies"
assert len(set(candidate_ids)) == len(candidate_ids)
assert Path(round_data["random"][2]["input_checkpoint"]) == Path(round_data["full"][2]["input_checkpoint"])
assert round_data["random"][2]["budget"] == round_data["full"][2]["budget"]
assert round_data["random"][2]["seed"] == round_data["full"][2]["seed"]
assert evaluation_split_path == str(Path(DATA_ROOT) / "ImageSets" / DATASET / f"{PILOT_EVAL_SPLIT}.txt")

random_dir = Path(LOCAL_RESULT_ROOT) / "random"
full_dir = Path(LOCAL_RESULT_ROOT) / "full"
assert not (random_dir / "proposal_scores.csv").exists()
assert not (random_dir / "image_scores.csv").exists()
assert not (random_dir / "reference_proposals.npz").exists()
assert (full_dir / "proposal_scores.csv").exists()
assert (full_dir / "image_scores.csv").exists()
assert (full_dir / "reference_proposals.npz").exists()

comparison_rows = []
selected_sets = {}
for strategy, (_, metrics, manifest, selected, _) in round_data.items():
    selected_sets[strategy] = set(selected)
    comparison_rows.append({
        "strategy": strategy,
        "selected_images": len(selected),
        "known_mAP": metrics["known_mAP"],
        "U_Recall": metrics["U_Recall"],
        "WI": metrics["WI"],
        "A_OSE": metrics["A_OSE"],
        "completed": manifest["completed"],
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison)
overlap_stats = {
    "overlap_count": len(selected_sets["random"] & selected_sets["full"]),
    "random_only_count": len(selected_sets["random"] - selected_sets["full"]),
    "full_only_count": len(selected_sets["full"] - selected_sets["random"]),
}
display(pd.DataFrame(overlap_stats.items(), columns=["stat", "value"]))
print("One seed and one round are an integration pilot, not evidence of scientific superiority.")


In [ ]:
def save_successful_results_to_drive():
    for strategy in STRATEGIES:
        manifest = json.loads((Path(LOCAL_RESULT_ROOT) / strategy / "round_manifest.json").read_text(encoding="utf-8"))
        if manifest.get("completed") is not True:
            raise RuntimeError(f"Refusing to copy incomplete {strategy} results to Drive.")
    pilot_summary = {
        "seed": SEED,
        "budget": BUDGET,
        "source_image_count": SOURCE_TASK2_IMAGES,
        "candidate_pool_count": len(candidate_ids),
        "reference_count": len(reference_ids),
        "evaluation_count": len(evaluation_ids),
        "imbalance_ratio": IMBALANCE_RATIO,
        "strategies": list(STRATEGIES),
        "metric_summary": comparison.to_dict(orient="records"),
        "overlap_statistics": overlap_stats,
        "DAOWOD_commit": repo_commits["DAOWOD"],
        "PROB_commit": repo_commits["PROB"],
        "benchmark_result": False,
    }
    summary_path = Path(LOCAL_RESULT_ROOT) / "pilot_summary.json"
    summary_path.write_text(json.dumps(pilot_summary, indent=2) + "\n", encoding="utf-8")
    destination = Path(DRIVE_RESULT_DIR)
    if destination.exists():
        shutil.rmtree(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_RESULT_ROOT, destination)
    if (destination / "data").exists() or (destination / "OWOD").exists():
        raise RuntimeError("Dataset extraction unexpectedly appeared in the Drive result directory.")
    display(pd.DataFrame({"saved_to": [str(destination)], "summary": [str(destination / "pilot_summary.json")]}))
    return pilot_summary

pilot_summary = run_step("Drive persistence", save_successful_results_to_drive)


In [ ]:
final_status = pd.DataFrame([{"stage": item, "status": STATUS[item]} for item in STATUS_ORDER])
display(final_status)
if not all(value == "OK" for value in STATUS.values()):
    raise RuntimeError("Pilot finished with one or more non-OK stages.")
print("Contribution A real active-round pilot completed successfully.")
